In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
import torch.nn as nn
import time
import warnings

warnings.filterwarnings('ignore')


ATTENTION_EPOCHS = 100
USE_ATTENTION_FOR_XGBOOST = False

class TrueDynamicAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim=None):
        super(TrueDynamicAttention, self).__init__()
        if hidden_dim is None:
            hidden_dim = max(input_dim // 2, 16)

        self.attention_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, input_dim),
            nn.Sigmoid()
        )

        self.global_importance = nn.Parameter(torch.ones(input_dim))

    def forward(self, x):
        sample_attention = self.attention_net(x)
        global_weight = torch.sigmoid(self.global_importance)
        final_attention = sample_attention * global_weight.unsqueeze(0)
        weighted_features = x * final_attention
        return weighted_features, final_attention


class DynamicAttentionWrapper:
    def __init__(self, input_dim, device='cpu', learning_rate=0.001):
        self.input_dim = input_dim
        self.device = device
        self.attention = TrueDynamicAttention(input_dim).to(device)
        self.is_fitted = False

    def fit(self, X, y, epochs=100, batch_size=256, verbose=True):
        self.attention.train()
        X_tensor = torch.FloatTensor(X).to(self.device)
        y_tensor = torch.LongTensor(y).to(self.device)

        n_classes = len(np.unique(y))
        classifier = nn.Linear(self.input_dim, n_classes).to(self.device)
        optimizer = torch.optim.Adam(
            list(self.attention.parameters()) + list(classifier.parameters()),
            lr=0.001
        )
        criterion = nn.CrossEntropyLoss()

        if verbose:
            print(f"  epoch: {epochs}")

        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

        best_loss = float('inf')
        patience_counter = 0

        for epoch in range(epochs):
            total_loss = 0
            for batch_X, batch_y in dataloader:
                optimizer.zero_grad()
                weighted_X, attention_weights = self.attention(batch_X)
                logits = classifier(weighted_X)
                loss = criterion(logits, batch_y)
                sparsity_loss = 0.01 * torch.mean(attention_weights)
                total_loss_batch = loss + sparsity_loss
                total_loss_batch.backward()
                optimizer.step()
                total_loss += loss.item()

            avg_loss = total_loss / len(dataloader)

            if avg_loss < best_loss:
                best_loss = avg_loss
                patience_counter = 0
            else:
                patience_counter += 1

            if verbose and (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1:3d}/{epochs}: Loss = {avg_loss:.4f}")

            if patience_counter >= 20:
                if verbose:
                    print(f"  Early stopping at epoch {epoch+1}")
                break

        self.is_fitted = True
        if verbose:
            print(f"Dynamic attention training completed!")
        return self

    def transform(self, X):
        self.attention.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            weighted_X, attention_weights = self.attention(X_tensor)
            return weighted_X.cpu().numpy(), attention_weights.cpu().numpy()

    def fit_transform(self, X, y, **kwargs):
        self.fit(X, y, **kwargs)
        return self.transform(X)


if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
    torch.cuda.set_device(0)
    device = 'cuda'
else:
    device = 'cpu'

print(f"\ndevice: {device.upper()}")


df = pd.read_csv('Esophageal_Dataset.csv')
target_col = 'stage_event_pathologic_stage'
df = df.dropna(subset=[target_col])

leakage_features = [
    'stage_event_tnm_categories',
    'stage_event_clinical_stage',
    'stage_event_system_version',
    'stage_event_psa',
    'stage_event_gleason_grading',
    'stage_event_ann_arbor',
    'stage_event_serum_markers',
    'stage_event_igcccg_stage',
    'stage_event_masaoka_stage',
    'vital_status',
    'days_to_death',
    'days_to_last_followup',
    'person_neoplasm_cancer_status',
    'has_new_tumor_events_information',
    'primary_pathology_number_of_lymphnodes_positive_by_he',
    'primary_pathology_number_of_lymphnodes_positive_by_ihc',
    'primary_pathology_lymph_node_metastasis_radiographic_evidence',
    'primary_pathology_lymph_node_examined_count',
    'primary_pathology_primary_lymph_node_presentation_assessment',
    'primary_pathology_residual_tumor',
    'primary_pathology_postoperative_rx_tx',
    'primary_pathology_radiation_therapy',
    'primary_pathology_planned_surgery_status',
    'primary_pathology_neoplasm_histologic_grade',
]

removed_leakage = [f for f in leakage_features if f in df.columns]
df = df.drop(columns=removed_leakage)


missing_rates = df.isnull().sum() / len(df)
high_missing = missing_rates[missing_rates > 0.8].index.tolist()
high_missing = [c for c in high_missing if c != target_col]
if high_missing:
    df = df.drop(columns=high_missing)
    print(f"{len(high_missing)} features with high missing rate have been removed")

exclude_kw = ['stage', 'unnamed', 'barcode', 'uuid', 'patient_id',
              'bcr_', 'icd_', 'project', 'has_', 'vital', 'days_to', 'diagnosis']

feature_cols = []
for col in df.columns:
    if col == target_col:
        continue
    if any(kw in col.lower() for kw in exclude_kw):
        continue
    if df[col].isnull().sum() / len(df) < 0.5:
        feature_cols.append(col)


X = df[feature_cols].copy()
y = df[target_col].copy()

for col in X.select_dtypes(include=[np.number]).columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        mode = X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown'
        X[col].fillna(mode, inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

for i, label in enumerate(le_target.classes_):
    print(f"  {label} → {i}")

print(f"Feature matrix: {X.shape}")

K_FEATURES = 15
selector = SelectKBest(
    score_func=lambda X, y: mutual_info_classif(X, y, random_state=42),
    k=min(K_FEATURES, len(feature_cols))
)
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'feature': feature_cols,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print(feature_scores.head(20).to_string(index=False))

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

for i, feat in enumerate(selected_features, 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:<65} (score: {score:.4f})")

X_selected = X[selected_features].copy()


class_counts = pd.Series(y_encoded).value_counts()
print("\nNumber of samples per stage:")
for stage_idx, count in class_counts.items():
    stage_name = le_target.classes_[stage_idx]
    print(f"  {stage_name} ({stage_idx}): {count}")

min_samples = class_counts.min()
stratify_param = None if min_samples < 2 else y_encoded

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=stratify_param
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("\nTraining set distribution (before SMOTE):")
for s, c in pd.Series(y_train).value_counts().sort_index().items():
    print(f"  {le_target.classes_[s]} ({s}): {c}")


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


dynamic_attention = DynamicAttentionWrapper(
    input_dim=X_train_scaled.shape[1],
    device=device,
    learning_rate=0.001
)

attention_start = time.time()
X_train_attended, train_attention_weights = dynamic_attention.fit_transform(
    X_train_scaled, y_train,
    epochs=ATTENTION_EPOCHS,
    batch_size=256,
    verbose=True
)
attention_time = time.time() - attention_start

X_test_attended, test_attention_weights = dynamic_attention.transform(X_test_scaled)

print(f"\nDynamic attention completed. Time: {attention_time:.2f}s")

print(f"\nAttention dynamics validation (first 3 features):")
for i in range(min(3, len(selected_features))):
    feat_weights = train_attention_weights[:, i]
    print(f"  Feature {i+1} ({selected_features[i]}): std = {feat_weights.std():.4f} (>0 indicates dynamic)")


smote = SMOTE(random_state=42)

X_train_tabnet, y_train_tabnet = smote.fit_resample(X_train_attended, y_train)
print(f"TabNet training set (dynamic attention): {X_train_tabnet.shape}")

if USE_ATTENTION_FOR_XGBOOST:
    X_train_xgb, y_train_xgb = X_train_tabnet, y_train_tabnet
    print(f"XGBoost training set (dynamic attention): {X_train_xgb.shape}")
else:
    X_train_xgb, y_train_xgb = smote.fit_resample(X_train_scaled, y_train)
    print(f"XGBoost training set (original data): {X_train_xgb.shape}")

print(f"\nBefore SMOTE: {X_train_scaled.shape[0]} samples")
print(f"After SMOTE - TabNet: {X_train_tabnet.shape[0]} samples")
print(f"After SMOTE - XGBoost: {X_train_xgb.shape[0]} samples")

print("\nTraining set distribution (after SMOTE):")
for s, c in pd.Series(y_train_tabnet).value_counts().sort_index().items():
    print(f"  {le_target.classes_[s]} ({s}): {c}")


tabnet_model = TabNetClassifier(
    n_d=32,
    n_a=32,
    n_steps=3,
    gamma=1.3,
    lambda_sparse=1e-3,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=0.02),
    scheduler_params={"step_size": 10, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax',
    seed=42,
    device_name=device,
    verbose=10
)



print("Training TabNet...")


start_time = time.time()

tabnet_model.fit(
    X_train_tabnet, y_train_tabnet,
    eval_set=[(X_test_attended, y_test)],
    eval_name=['validation'],
    eval_metric=['accuracy'],
    max_epochs=100,
    patience=20,
    batch_size=256,
    virtual_batch_size=128,
    drop_last=False
)

training_time = time.time() - start_time

print(f"TabNet training completed")
print(f"Total training time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print(f"Actual training epochs: {len(tabnet_model.history['loss'])}")


if USE_ATTENTION_FOR_XGBOOST:
    xgb_test_data = X_test_attended
else:
    xgb_test_data = X_test_scaled

eval_set = [(X_train_xgb, y_train_xgb), (xgb_test_data, y_test)]
eval_names = ['train', 'validation']

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.01,
    max_depth=5,
    subsample=1.0,
    colsample_bytree=1.0,
    objective='multi:softprob',
    num_class=len(np.unique(y_train_xgb)),
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)


print("Training XGBoost...")


start_time_xgb = time.time()

xgb_model.fit(
    X_train_xgb, y_train_xgb,
    eval_set=eval_set,
    verbose=10
)

xgb_training_time = time.time() - start_time_xgb

print(f"XGBoost training completed")
print(f"Training time: {xgb_training_time:.2f} seconds ({xgb_training_time/60:.2f} minutes)")

evals_result = xgb_model.evals_result()
train_logloss = evals_result['validation_0']['mlogloss']
val_logloss = evals_result['validation_1']['mlogloss']

best_iteration = np.argmin(val_logloss) + 1


tabnet_train_proba = tabnet_model.predict_proba(X_train_tabnet)

if USE_ATTENTION_FOR_XGBOOST:
    xgb_train_proba = xgb_model.predict_proba(X_train_tabnet)
else:
    xgb_train_proba = xgb_model.predict_proba(X_train_xgb)

print(f"  TabNet output shape: {tabnet_train_proba.shape}")
print(f"  XGBoost output shape: {xgb_train_proba.shape}")

meta_train_features = np.hstack([tabnet_train_proba, xgb_train_proba])
print(f"  Meta-learner input shape: {meta_train_features.shape}")

meta_learner = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42,
    multi_class='multinomial',
    solver='lbfgs',
    verbose=1
)

start_time_meta = time.time()

meta_learner.fit(meta_train_features, y_train_tabnet)

meta_training_time = time.time() - start_time_meta

print(f"Meta-learner training completed")
print(f"  Training time: {meta_training_time:.2f} seconds")
print(f"  Number of iterations: {meta_learner.n_iter_[0]}")


from sklearn.base import BaseEstimator, ClassifierMixin

class StackedEnsemble(BaseEstimator, ClassifierMixin):

    def __init__(self, device='cpu', random_state=42):
        self.device = device
        self.random_state = random_state

    def fit(self, X, y):
        X_arr = X.values if hasattr(X, 'values') else X

        scaler_ = StandardScaler()
        X_scaled = scaler_.fit_transform(X_arr)
        self.scaler_ = scaler_

        attn = DynamicAttentionWrapper(input_dim=X_scaled.shape[1], device=self.device)
        X_attended, _ = attn.fit_transform(X_scaled, y, epochs=ATTENTION_EPOCHS,
                                           batch_size=256, verbose=False)
        self.attn_ = attn

        smote_ = SMOTE(random_state=self.random_state)
        X_tabnet, y_tabnet = smote_.fit_resample(X_attended, y)

        if USE_ATTENTION_FOR_XGBOOST:
            X_xgb, y_xgb = X_tabnet, y_tabnet
        else:
            X_xgb, y_xgb = smote_.fit_resample(X_scaled, y)

        self.tabnet = TabNetClassifier(
            n_d=32, n_a=32, n_steps=3, gamma=1.3, lambda_sparse=1e-3,
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=0.02),
            seed=self.random_state,
            device_name=self.device,
            verbose=0
        )
        self.tabnet.fit(
            X_tabnet, y_tabnet,
            max_epochs=100, patience=20, batch_size=256,
            virtual_batch_size=128
        )

        self.xgb = xgb.XGBClassifier(
            n_estimators=200, learning_rate=0.01, max_depth=5,
            subsample=1.0, objective='multi:softprob',
            num_class=len(np.unique(y)),
            random_state=self.random_state, verbosity=0
        )
        self.xgb.fit(X_xgb, y_xgb)

        tabnet_proba = self.tabnet.predict_proba(X_tabnet)
        if USE_ATTENTION_FOR_XGBOOST:
            xgb_proba = self.xgb.predict_proba(X_tabnet)
        else:
            xgb_proba = self.xgb.predict_proba(X_xgb)
        meta_features = np.hstack([tabnet_proba, xgb_proba])

        self.meta = LogisticRegression(
            C=1.0, max_iter=1000, random_state=self.random_state,
            multi_class='multinomial', solver='lbfgs', verbose=0
        )
        self.meta.fit(meta_features, y_tabnet)

        return self

    def predict(self, X):
        X_arr = X.values if hasattr(X, 'values') else X
        X_scaled = self.scaler_.transform(X_arr)
        X_attended, _ = self.attn_.transform(X_scaled)

        tabnet_proba = self.tabnet.predict_proba(X_attended)
        if USE_ATTENTION_FOR_XGBOOST:
            xgb_proba = self.xgb.predict_proba(X_attended)
        else:
            xgb_proba = self.xgb.predict_proba(X_scaled)
        meta_features = np.hstack([tabnet_proba, xgb_proba])
        return self.meta.predict(meta_features)


ensemble_cv = StackedEnsemble(device=device, random_state=42)

cv_start_time = time.time()
cv_scores = cross_val_score(
    ensemble_cv, X_selected, y_encoded,
    cv=5, scoring='accuracy', n_jobs=1
)
cv_time = time.time() - cv_start_time

for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i:2d}: {score:.4f}")
print(f"\n{'='*40}")
print(f"  Average accuracy: {cv_scores.mean():.4f}")
print(f"  Standard deviation: {cv_scores.std():.4f}")
print(f"  Confidence interval: [{cv_scores.mean()-1.96*cv_scores.std():.4f}, {cv_scores.mean()+1.96*cv_scores.std():.4f}]")


xgb_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nXGBoost:")
print(xgb_importance.to_string(index=False))

tabnet_feature_importance = tabnet_model.feature_importances_
tabnet_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': tabnet_feature_importance
}).sort_values('importance', ascending=False)

print("TabNet:")
print(tabnet_importance.to_string(index=False))

avg_attention = train_attention_weights.mean(axis=0)
attention_importance = pd.DataFrame({
    'feature': selected_features,
    'attention_weight': avg_attention
}).sort_values('attention_weight', ascending=False)

print("Dynamic Attention (average weight):")
print(attention_importance.to_string(index=False))


tabnet_test_proba = tabnet_model.predict_proba(X_test_attended)

if USE_ATTENTION_FOR_XGBOOST:
    xgb_test_proba = xgb_model.predict_proba(X_test_attended)
else:
    xgb_test_proba = xgb_model.predict_proba(X_test_scaled)

meta_test_features = np.hstack([tabnet_test_proba, xgb_test_proba])
y_test_pred = meta_learner.predict(meta_test_features)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_recall = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

tabnet_train_orig_proba = tabnet_model.predict_proba(X_train_attended)

if USE_ATTENTION_FOR_XGBOOST:
    xgb_train_orig_proba = xgb_model.predict_proba(X_train_attended)
else:
    xgb_train_orig_proba = xgb_model.predict_proba(X_train_scaled)

meta_train_orig = np.hstack([tabnet_train_orig_proba, xgb_train_orig_proba])
y_train_pred = meta_learner.predict(meta_train_orig)

train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_recall = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1 = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)


print("\n" + "=" * 70)
print(" " * 20 + "Model Performance Comparison")
print("=" * 70)
print(f"{'Metric':<15} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 70)
print(f"{'Accuracy':<15} {train_accuracy:>15.4f} {test_accuracy:>15.4f} {abs(train_accuracy-test_accuracy):>15.4f}")
print(f"{'Precision':<15} {train_precision:>15.4f} {test_precision:>15.4f} {abs(train_precision-test_precision):>15.4f}")
print(f"{'Recall':<15} {train_recall:>15.4f} {test_recall:>15.4f} {abs(train_recall-test_recall):>15.4f}")
print(f"{'F1 Score':<15} {train_f1:>15.4f} {test_f1:>15.4f} {abs(train_f1-test_f1):>15.4f}")
print("=" * 70)

print("=" * 70)
print(classification_report(
    y_test, y_test_pred,
    target_names=[str(c) for c in le_target.classes_],
    zero_division=0
))

